In [31]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [32]:
from backend.ai.rag_pipeline import get_rag_response

In [33]:
question = "How can I reset my password?"

response = get_rag_response(question)

print(response)

Toreset your password, follow these steps:

1. Go to the login page of the service or app.  
2. Click the **“Forgot password?”** or **“Reset password”** link (usually near the password field).  
3. Enter the email address associated with your account and submit the request.  
4. Check your email inbox (and spam/junk folder) for a password‑reset message.  
5. Click the reset link in the email and follow the prompts to create a new password.  
6. Log in with your new password.

If you don’t receive the email or encounter any issues, please let me know the specific platform you’re trying to access, and I can provide more detailed instructions or escalate the request to support.


In [34]:
import pandas as pd

test_data = pd.read_csv("../data/evaluation/test_dataset.csv")

test_data

,question,ground_truth
0,How can I reset my password?,"To reset your password, click 'Forgot Password..."
1,How do I cancel my order?,You can cancel your order before it is shipped...
2,How can I request a refund?,You can request a refund by contacting custome...
3,How do I track my order?,You can track your order using the tracking nu...
4,How do I change my email address?,Go to Account Settings and update your email a...
5,How can I contact customer support?,You can contact customer support through live ...
6,What payment methods do you accept?,"We accept credit cards, debit cards, and PayPal."
7,How long does shipping take?,Shipping usually takes between 3 and 7 busines...
8,Can I change my delivery address?,You can change your delivery address before yo...
9,How do I update my account information?,Go to your profile settings and update your pe...


In [35]:
predictions = []

for question in test_data["question"]:
    answer = get_rag_response(question)
    predictions.append(answer)

test_data["prediction"] = predictions

test_data

,question,ground_truth,prediction
0,How can I reset my password?,"To reset your password, click 'Forgot Password...","Toreset your password, follow these steps:\n\n..."
1,How do I cancel my order?,You can cancel your order before it is shipped...,"To cancel your order, please follow these gene..."
2,How can I request a refund?,You can request a refund by contacting custome...,"Okay, the user is asking how to request a refu..."
3,How do I track my order?,You can track your order using the tracking nu...,"To track your order, you'll typically need you..."
4,How do I change my email address?,Go to Account Settings and update your email a...,1. Log in to your account on the website or ap...
5,How can I contact customer support?,You can contact customer support through live ...,You can typically contact customer support thr...
6,What payment methods do you accept?,"We accept credit cards, debit cards, and PayPal.",We accept the following payment methods:\n\n- ...
7,How long does shipping take?,Shipping usually takes between 3 and 7 busines...,Shipping times vary depending on your location...
8,Can I change my delivery address?,You can change your delivery address before yo...,"Yes, you can often change your delivery addres..."
9,How do I update my account information?,Go to your profile settings and update your pe...,Here are the general steps to update your acco...


In [44]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

bleu_scores = []

for _, row in test_data.iterrows():
    score = sentence_bleu(
        [row["ground_truth"].split()],
        row["prediction"].split(),
        smoothing_function=smooth
    )

    bleu_scores.append(score)

test_data["BLEU"] = bleu_scores

print("Average BLEU:", sum(bleu_scores) / len(bleu_scores))

Average BLEU: 0.007041957930289638


In [45]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1 = []
rouge2 = []
rougeL = []

for _, row in test_data.iterrows():

    scores = scorer.score(
        row["ground_truth"],
        row["prediction"]
    )

    rouge1.append(scores["rouge1"].fmeasure)
    rouge2.append(scores["rouge2"].fmeasure)
    rougeL.append(scores["rougeL"].fmeasure)

test_data["ROUGE-1"] = rouge1
test_data["ROUGE-2"] = rouge2
test_data["ROUGE-L"] = rougeL

print("Average ROUGE-1:", sum(rouge1)/len(rouge1))
print("Average ROUGE-2:", sum(rouge2)/len(rouge2))
print("Average ROUGE-L:", sum(rougeL)/len(rougeL))

Average ROUGE-1: 0.1112873183779935
Average ROUGE-2: 0.03912004979266917
Average ROUGE-L: 0.08947836126115082


In [46]:
test_data

,question,ground_truth,prediction,BLEU,ROUGE-1,ROUGE-2,ROUGE-L
0,How can I reset my password?,"To reset your password, click 'Forgot Password...","Toreset your password, follow these steps:\n\n...",0.016264,0.220472,0.096000,0.157480
1,How do I cancel my order?,You can cancel your order before it is shipped...,"To cancel your order, please follow these gene...",0.006406,0.157895,0.053571,0.122807
2,How can I request a refund?,You can request a refund by contacting custome...,"Okay, the user is asking how to request a refu...",0.001035,0.030197,0.011641,0.025552
3,How do I track my order?,You can track your order using the tracking nu...,"To track your order, you'll typically need you...",0.008055,0.203704,0.075472,0.148148
4,How do I change my email address?,Go to Account Settings and update your email a...,1. Log in to your account on the website or ap...,0.005487,0.139130,0.070796,0.139130
5,How can I contact customer support?,You can contact customer support through live ...,You can typically contact customer support thr...,0.020339,0.111111,0.056338,0.111111
6,What payment methods do you accept?,"We accept credit cards, debit cards, and PayPal.",We accept the following payment methods:\n\n- ...,0.009570,0.301887,0.078431,0.188679
7,How long does shipping take?,Shipping usually takes between 3 and 7 busines...,Shipping times vary depending on your location...,0.002218,0.050847,0.000000,0.033898
8,Can I change my delivery address?,You can change your delivery address before yo...,"Yes, you can often change your delivery addres...",0.016838,0.204545,0.116279,0.204545
9,How do I update my account information?,Go to your profile settings and update your pe...,Here are the general steps to update your acco...,0.004786,0.132353,0.044776,0.088235


In [47]:
import mlflow

In [48]:
mlflow.set_experiment("RAG Evaluation")

with mlflow.start_run():
    print("MLflow Run Started")


MLflow Run Started


In [ ]:
mlflow.log_param("embedding_model", "sentence-transformers/all-MiniLM-L6-v2")
mlflow.log_param("retrieval_k", 5)
mlflow.log_param("relevance_threshold", 0.25) 

0.25

In [50]:
mlflow.log_metric("BLEU", sum(bleu_scores) / len(bleu_scores))
mlflow.log_metric("ROUGE-1", sum(rouge1) / len(rouge1))
mlflow.log_metric("ROUGE-2", sum(rouge2) / len(rouge2))
mlflow.log_metric("ROUGE-L", sum(rougeL) / len(rougeL))

In [ ]:
mlflow.end_run() 

print("Experiment Saved Successfully!")

Experiment Saved Successfully!
